In [30]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
import pandas as pd
import yfinance as yf
from arch import arch_model

import seaborn as sns
import matplotlib.pyplot as plt

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource

In [32]:
RISKY_ASSET = "^GSPC"
START_DATE = "2018-01-01"
END_DATE = "2024-10-19"

In [33]:
df = yf.download(RISKY_ASSET,
                 start=START_DATE,
                 end=END_DATE)['Adj Close']

[*********************100%***********************]  1 of 1 completed


In [34]:
returns = 100 * df.pct_change().dropna()
returns.name = "asset_returns"
print(f"Average return: {round(returns.mean(), 2)}%")

Average return: 0.05%


In [35]:

output_notebook()
source = ColumnDataSource(data=dict(date=returns.index, returns=returns))

p = figure(title=f"{RISKY_ASSET} returns: {START_DATE} - {END_DATE}", x_axis_label='Date', y_axis_label='Returns', x_axis_type='datetime')
p.line('date', 'returns', source=source, line_width=2)

show(p)

Loading BokehJS ...

In [36]:
model = arch_model(returns, mean="Zero", vol="ARCH", p=1, q=0)

In [37]:
fitted_model = model.fit(disp="off")
print(fitted_model.summary())

                        Zero Mean - ARCH Model Results                        
Dep. Variable:          asset_returns   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.001
Vol Model:                       ARCH   Log-Likelihood:               -2595.07
Distribution:                  Normal   AIC:                           5194.14
Method:            Maximum Likelihood   BIC:                           5205.03
                                        No. Observations:                 1710
Date:                Sat, Nov 02 2024   Df Residuals:                     1710
Time:                        15:57:16   Df Model:                            0
                            Volatility Model                            
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
omega          0.8435  6.241e-02     13.516  1.257e-41 [  0.721,  0.96

In [38]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource

# Create a ColumnDataSource from the fitted model's residuals
residuals = fitted_model.resid
source_residuals = ColumnDataSource(data=dict(date=returns.index, residuals=residuals))

# Create a new figure for the residuals plot
p_residuals = figure(title="Residuals of the Fitted Model", x_axis_label='Date', y_axis_label='Residuals', x_axis_type='datetime')
p_residuals.line('date', 'residuals', source=source_residuals, line_width=2)

#create a new graph of annualized conditional volatility
annualized_volatility = 100 * fitted_model.conditional_volatility
source_volatility = ColumnDataSource(data=dict(date=returns.index, volatility=annualized_volatility))

# Create a new figure for the annualized volatility plot
p_volatility = figure(title="Annualized Conditional Volatility", x_axis_label='Date', y_axis_label='Volatility', x_axis_type='datetime')
p_volatility.line('date', 'volatility', source=source_volatility, line_width=2)

# Show the plot
show(p_residuals)
show(p_volatility)

In [39]:
diagnostics_dict = {
    "resids": fitted_model.resid,
    "conditional_volatility": fitted_model.conditional_volatility,
    "std_resid": fitted_model.std_resid,
    "std_resid_manual": fitted_model.resid / fitted_model.conditional_volatility,
}

df_diagnostics = pd.DataFrame(data = diagnostics_dict)
df_diagnostics

,resids,conditional_volatility,std_resid,std_resid_manual
Date,,,,
2018-01-03,0.639882,1.145349,0.558678,0.558678
2018-01-04,0.402864,1.014721,0.397019,0.397019
2018-01-05,0.703377,0.957755,0.734402,0.734402
2018-01-08,0.166234,1.033652,0.160822,0.160822
2018-01-09,0.130293,0.925240,0.140821,0.140821
...,...,...,...,...
2024-10-14,0.770767,1.004997,0.766934,0.766934
2024-10-15,-0.760947,1.055273,-0.721090,-0.721090
2024-10-16,0.467915,1.052028,0.444774,0.444774


In [40]:
#Test the residuals of the ARCH(1) model with the LM test.
from statsmodels.stats.diagnostic import het_arch
het_arch(fitted_model.resid)

(np.float64(677.5732104292927),
 np.float64(4.087739828440285e-139),
 111.93184334455775,
 1.8061420127596109e-178)

In [41]:
#As the residuals come from a model in which we estimated two parameters (omega and alpha), 
# we should correct for that when using the het_arch test.

In [42]:
het_arch(fitted_model.resid, ddof=2)

(np.float64(676.7760654758465),
 np.float64(6.0609710154554375e-139),
 111.93184334455775,
 1.8061420127596109e-178)

In [43]:
#Modeling stock returns' volatility with GARCH models

In [44]:
model = arch_model(returns, mean="Zero", vol="GARCH", p=1, q=1)

In [45]:
#Estimate the model and print the summary
fitted_model = model.fit(disp="off")
print(fitted_model.summary())


                       Zero Mean - GARCH Model Results                        
Dep. Variable:          asset_returns   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.001
Vol Model:                      GARCH   Log-Likelihood:               -2391.43
Distribution:                  Normal   AIC:                           4788.87
Method:            Maximum Likelihood   BIC:                           4805.20
                                        No. Observations:                 1710
Date:                Sat, Nov 02 2024   Df Residuals:                     1710
Time:                        15:57:36   Df Model:                            0
                              Volatility Model                              
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omega          0.0436  1.367e-02      3.190  1.423e-03 [1.

In [46]:
# Create a ColumnDataSource from the fitted model's residuals
residuals = fitted_model.resid
source_residuals = ColumnDataSource(data=dict(date=returns.index, residuals=residuals))

# Create a new figure for the residuals plot
p_residuals = figure(title="Residuals of the Fitted Model", x_axis_label='Date', y_axis_label='Residuals', x_axis_type='datetime')
p_residuals.line('date', 'residuals', source=source_residuals, line_width=2)

#create a new graph of annualized conditional volatility
annualized_volatility = 100 * fitted_model.conditional_volatility
source_volatility = ColumnDataSource(data=dict(date=returns.index, volatility=annualized_volatility))

# Create a new figure for the annualized volatility plot
p_volatility = figure(title="Annualized Conditional Volatility", x_axis_label='Date', y_axis_label='Volatility', x_axis_type='datetime')
p_volatility.line('date', 'volatility', source=source_volatility, line_width=2)

# Show the plot
show(p_residuals)
show(p_volatility)

In [57]:
#Forecasting volatility using GARCH models

from datetime import datetime

In [58]:
returns = 100 * df.pct_change().dropna()
returns.name = "asset_returns"

In [59]:
model = arch_model(returns, mean="Zero", vol="GARCH", dist="t",p=1, q=1)

In [60]:
#Define the split date and fit the model:
SPLIT_DATE = datetime(2023, 1, 1)
fitted_model = model.fit(last_obs=SPLIT_DATE, disp="off")

In [61]:
#Create and inspect the analytical forecasts:

forecasts_analytical = fitted_model.forecast(horizon=3, 
                                             start=SPLIT_DATE,
                                             reindex=False)

# Prepare data for Bokeh plot
forecast_dates = forecasts_analytical.variance.index
forecast_variances = forecasts_analytical.variance.values.T

source_forecast = ColumnDataSource(data=dict(
    date=forecast_dates,
    variance_1=forecast_variances[0],
    variance_2=forecast_variances[1],
    variance_3=forecast_variances[2]
))

# Create a new figure for the forecast plot
p_forecast = figure(title="Analytical forecasts for different horizons", x_axis_label='Date', y_axis_label='Variance', x_axis_type='datetime')

# Add lines for each forecast horizon
p_forecast.line('date', 'variance_1', source=source_forecast, line_width=2, color='blue', legend_label='Horizon 1')
p_forecast.line('date', 'variance_2', source=source_forecast, line_width=2, color='green', legend_label='Horizon 2')
p_forecast.line('date', 'variance_3', source=source_forecast, line_width=2, color='red', legend_label='Horizon 3')

# Show the plot
show(p_forecast)

In [62]:
forecasts_analytical.variance

,h.1,h.2,h.3
Date,,,
2023-01-03,1.211590,1.240041,1.268428
2023-01-04,1.117560,1.146224,1.174823
2023-01-05,1.191662,1.220158,1.248589
2023-01-06,1.986648,2.013347,2.039986
2023-01-09,1.636180,1.663671,1.691101
...,...,...,...
2024-10-14,0.623881,0.653661,0.683373
2024-10-15,0.645122,0.674854,0.704518
2024-10-16,0.593703,0.623551,0.653331


In [63]:
#Create and inspect the simulation forecasts

forecasts_simulation = fitted_model.forecast(horizon=3, start=SPLIT_DATE, method='simulation', reindex=False)

# Prepare data for Bokeh plot
forecast_dates_sim = forecasts_simulation.variance.index
forecast_variances_sim = forecasts_simulation.variance.values.T

source_forecast_sim = ColumnDataSource(data=dict(
    date=forecast_dates_sim,
    variance_1=forecast_variances_sim[0],
    variance_2=forecast_variances_sim[1],
    variance_3=forecast_variances_sim[2]
))

# Create a new figure for the forecast plot
p_forecast_sim = figure(title="Simulation forecasts for different horizons", x_axis_label='Date', y_axis_label='Variance', x_axis_type='datetime')

# Add lines for each forecast horizon
p_forecast_sim.line('date', 'variance_1', source=source_forecast_sim, line_width=2, color='blue', legend_label='Horizon 1')
p_forecast_sim.line('date', 'variance_2', source=source_forecast_sim, line_width=2, color='green', legend_label='Horizon 2')
p_forecast_sim.line('date', 'variance_3', source=source_forecast_sim, line_width=2, color='red', legend_label='Horizon 3')

# Show the plot
show(p_forecast_sim)


In [64]:

forecasts_simulation.variance

,h.1,h.2,h.3
Date,,,
2023-01-03,1.211590,1.228556,1.228018
2023-01-04,1.117560,1.139992,1.162664
2023-01-05,1.191662,1.222314,1.265659
2023-01-06,1.986648,2.013890,2.068806
2023-01-09,1.636180,1.645348,1.683258
...,...,...,...
2024-10-14,0.623881,0.661880,0.704547
2024-10-15,0.645122,0.671197,0.697783
2024-10-16,0.593703,0.633997,0.664693


In [65]:
#Create and inspect the bootstrap forecasts

forecasts_bootstrap = fitted_model.forecast(horizon=3, start=SPLIT_DATE, method='bootstrap', reindex=False)

# Prepare data for Bokeh plot
forecast_dates_bootstrap = forecasts_bootstrap.variance.index
forecast_variances_bootstrap = forecasts_bootstrap.variance.values.T

source_forecast_bootstrap = ColumnDataSource(data=dict(
    date=forecast_dates_bootstrap,
    variance_1=forecast_variances_bootstrap[0],
    variance_2=forecast_variances_bootstrap[1],
    variance_3=forecast_variances_bootstrap[2]
))

# Create a new figure for the forecast plot
p_forecast_bootstrap = figure(title="Bootstrap forecasts for different horizons", x_axis_label='Date', y_axis_label='Variance', x_axis_type='datetime')

# Add lines for each forecast horizon
p_forecast_bootstrap.line('date', 'variance_1', source=source_forecast_bootstrap, line_width=2, color='blue', legend_label='Horizon 1')
p_forecast_bootstrap.line('date', 'variance_2', source=source_forecast_bootstrap, line_width=2, color='green', legend_label='Horizon 2')
p_forecast_bootstrap.line('date', 'variance_3', source=source_forecast_bootstrap, line_width=2, color='red', legend_label='Horizon 3')

# Show the plot
show(p_forecast_bootstrap)
